# Les 9 - Images

28/04/2026

## Herhalingsoefening
Schrijf een Python-script dat de "Picture of the Day" van Wikipedia ophaalt voor een door de gebruiker ingegeven datum.

Wat moet je script doen?

- Datum inlezen Vraag de gebruiker om een dag, maand en jaar in te geven.
- URL opbouwen De POTD-pagina van Wikipedia heeft dit formaat: https://en.wikipedia.org/wiki/Wikipedia:Picture_of_the_day Zoek uit hoe je naar een andere datum kan. Gebruik de datetime-module om de datum correct te formatteren en verwerk ze tot een geldige URL.
- Pagina scrapen Haal de pagina op met requests en gebruik BeautifulSoup om de grote afbeelding te vinden. Inspecteer de pagina eerst zelf via F12 in je browser.
- Afbeelding downloaden Download de afbeelding en sla ze op in een map output/ met een duidelijke bestandsnaam die de datum bevat.

In [11]:
import requests
from bs4 import BeautifulSoup

dag = input("Geef de dag in")
maand = input("Geef de maand in")
jaar = input("Geef het jaar in")

url = f"https://en.wikipedia.org/wiki/Template:POTD/{jaar}-{maand}-{dag}"

# print(url)

headers={'User-Agent': 'Mozilla/5.0'} #doen alsof je een browser bent
response = requests.get(url, headers=headers)

soup = BeautifulSoup(response.text)
div = soup.find("div", attrs={"id":"mw-content-text"})
img = div.find("img")
src = img["src"]

response_img = requests.get("https:"+src,headers=headers)
# print(response_img.status_code)
# print(response_img.url)
# Create image-file --> 'wb' = write binary mode (essential for non-text files like JPG/PNG)
with open(f"img/{jaar}-{maand}-{dag}.jpg", 'wb') as file:
    file.write(response_img.content) # .content is used for raw bytes

## Images - Pillow
Package Pillow

In [13]:
! py -m pip install Pillow


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [18]:
from PIL import ImageColor

ImageColor.getcolor('white',"RGBA")

(255, 255, 255, 255)

In [ ]:
from PIL import Image

img = Image.open("img/2026-04-27.jpg")
print(img)
img.show()
print(img.size)
width, height = img.size

img_new = Image.new("RGBA",(100,100),"pink")
img_new.show()
img_new.save("pink.png")

half_img = img.resize((int(width/2), int(height/2)))
half_img.rotate(180).save("rotate.png")
half_img.show()

<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=500x333 at 0x18DD953A0D0>
(500, 333)


In [ ]:
from PIL import Image

img = Image.open("img/2026-04-27.jpg")

for x in range(0, img.width):
    for y in range(0,img.height):
        r,g,b = img.getpixel((x,y))
        img.putpixel((x,y),(255-r, 255-g, 255-b))

img.save("inverted.png")

In [40]:
from PIL import Image

img = Image.open("img/2026-04-27.jpg")

for x in range(0, img.width):
    for y in range(0,img.height):
        r,g,b = img.getpixel((x,y))
        if r > 128 and g > 128 and b > 128:
            img.putpixel((x,y),(255,255,255))
        else:
            img.putpixel((x,y),(0,0,0))

img.save("blackwhite.png")

In [41]:
from PIL import Image

img = Image.open("img/2026-04-27.jpg")

for x in range(0, img.width):
    for y in range(0,img.height):
        r,g,b = img.getpixel((x,y))
        img.putpixel((x,y),(r,0,b))

img.save("nogreen.png")

## Images - opencv

https://docs.opencv.org/4.x/d6/d00/tutorial_py_root.html

In [43]:
! py -m pip install opencv-python


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import cv2
import numpy as np

img = cv2.imread("img/2026-04-27.jpg")

grey = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

cv2.imwrite("grey.jpg", grey)

True

In [46]:
import cv2
import numpy as np

# Load image and convert to grayscale
img = cv2.imread("img/2026-04-27.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Apply threshold to get binary image
ret, thresh = cv2.threshold(gray, 127, 255, 0)

# Find contours
contours, hierarchy = cv2.findContours(thresh, 
                               cv2.RETR_TREE, 
                               cv2.CHAIN_APPROX_SIMPLE)

# Draw contours on original image
contour_img = img.copy()
cv2.drawContours(contour_img, contours, -1, (0, 255, 0), 2)

cv2.imwrite("img_contours.jpg", contour_img)

True

In [49]:
import cv2

# Load image and convert to grayscale
img = cv2.imread("klas.jpeg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# Simple edge detection with Canny
edges = cv2.Canny(gray, 100, 200)

# Save result
cv2.imwrite('klas_edges.png', edges)

True

## YOLO

In [51]:
! py -m pip install ultralytics


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [54]:
from ultralytics import YOLO

# Load a pre-trained model (small version)
model = YOLO("yolov8n.pt")

# Perform detection on an image
image_results = model("klas.jpeg", save=True)



image 1/1 c:\Users\u0116591\OneDrive - Thomas More\GitHub 24-25\Scripting-Students\07 Images\klas.jpeg: 384x640 11 persons, 1 bottle, 8 chairs, 1 dining table, 9 laptops, 67.9ms
Speed: 3.3ms preprocess, 67.9ms inference, 10.0ms postprocess per image at shape (1, 3, 384, 640)
Results saved to C:\Users\u0116591\OneDrive - Thomas More\GitHub 24-25\Scripting-Students\07 Images\runs\detect\predict2


In [55]:
from ultralytics import YOLO
import cv2

model = YOLO("yolov8n.pt")
cap = cv2.VideoCapture(0)

print("Press 'q' to stop")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    # Detectie
    results = model(frame, verbose=False)
    annotated_frame = results[0].plot()
    
    # Tel objecten
    num_objects = len(results[0].boxes)
    
    # Voeg tekst toe
    cv2.putText(annotated_frame, f'Objecten: {num_objects}', 
                (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2)
    
    cv2.imshow('AI Webcam Demo', annotated_frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

Press 'q' to stop
